# EP1 - Agente veterinario: demo de extremo a extremo

Pipeline del agente de apoyo clinico (5 pasos):

1. Recuperar la **ficha del paciente** (RAG interno).
2. Recuperar la **entrada de dosificacion** especie + farmaco (RAG interno).
3. **Calcular el rango mg/kg x peso** con la tool de dosis.
4. **Verificar interacciones** contra los medicamentos actuales.
5. Formular la respuesta con citas [F#]/[T#] y aplicar **guardrails post-LLM**.

Para reproducibilidad la demo usa `ClienteFalso` (determinista, sin cuota Groq).
Para usar el LLM real: `ClienteFalso()` -> `ClienteGroq()` (modelo `openai/gpt-oss-120b`).

## Setup

In [ ]:
import sys
from pathlib import Path

# Raiz del repo (funciona abra el notebook desde donde lo abra)
RAIZ = Path().resolve()
if RAIZ.name == "notebooks":
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))
import os
os.chdir(RAIZ)

from agent.llm_client import ClienteFalso
from agent.reasoning_loop import AgenteVeterinario
from agent.trace import Trazador

trazador = Trazador(RAIZ / "logs/trace_demo.ipynb.jsonl")
agente = AgenteVeterinario(llm=ClienteFalso(), trazador=trazador)
print("Agente listo con ClienteFalso y trace en logs/trace_demo.ipynb.jsonl")

## Caso 1: interaccion severa detectada

Firulais (FIC-001, perro 24.5 kg) ya toma **meloxicam**; se propone **carprofeno** (dos AINEs). El guardrail antepone la alerta a la respuesta.

In [ ]:
r1 = agente.planificar(
    "perro con dolor articular",
    especie="perro", peso_kg=24.5, farmaco="carprofeno", paciente="FIC-001",
)
print(r1.texto)
print()
print("Ficha:", r1.ficha_id, "| Entrada:", r1.entrada_id)
print("Dosis:", r1.dosis.dosis_min_mg, "-", r1.dosis.dosis_max_mg, "mg")
print("Guardrails:", r1.guardrails)

## Caso 2: dosis calculada sin alertas

Luna (FIC-002, gata 4.2 kg) sin medicamentos actuales; se propone **meloxicam** felino.

In [ ]:
r2 = agente.planificar(
    "gata con dolor leve",
    especie="gato", peso_kg=4.2, farmaco="meloxicam", paciente="FIC-002",
)
print(r2.texto)
print()
print("Ficha:", r2.ficha_id, "| Entrada:", r2.entrada_id)
print("Dosis:", r2.dosis.dosis_min_mg, "-", r2.dosis.dosis_max_mg, "mg")
print("Alertas:", r2.alertas or "ninguna")
print("Fuentes:", r2.fuentes_citadas)

## Caso 3: guardrail sin informacion (negativa honesta)

Michi (FIC-005, gato 5.8 kg) y **carprofeno**: la guia interna NO tiene la combinacion carprofeno-gato (hueco deliberado del dataset). El guardrail REEMPLAZA el texto del LLM: ninguna cifra puede filtrarse.

In [ ]:
r3 = agente.planificar(
    "gato con dolor articular",
    especie="gato", peso_kg=5.8, farmaco="carprofeno", paciente="FIC-005",
)
print(r3.texto)
print()
print("sin_informacion:", r3.sin_informacion, "| entrada_id:", r3.entrada_id)
print("Guardrails:", r3.guardrails)
print("La respuesta contiene 'mg'?", "mg" in r3.texto)

## Caso 4: interaccion moderada sin bloqueo

Bonito (FIC-006, conejo 1.8 kg) ya toma **enrofloxacina**; se propone **gentamicina**: interaccion registrada (moderada) pero el farmaco no existe en la guia para conejo -> negativa con alerta visible.

In [ ]:
r4 = agente.planificar(
    "conejo con infeccion severa",
    especie="conejo", peso_kg=1.8, farmaco="gentamicina", paciente="FIC-006",
)
print(r4.texto)
print()
print("sin_informacion:", r4.sin_informacion, "| severa:", r4.alerta_severa)
print("Alertas:", r4.alertas)

## Trazabilidad de la corrida

Cada paso del loop queda en el JSONL: busquedas RAG, calculo de dosis, verificacion de interacciones, guardrails y respuesta final.

In [ ]:
import json

log_path = RAIZ / "logs/trace_demo.ipynb.jsonl"
lineas = log_path.read_text(encoding="utf-8").strip().split("\n")
for linea in lineas[-14:]:
    e = json.loads(linea)
    extras = {k: v for k, v in e.items() if k not in ("paso", "tipo", "hora_utc")}
    print(f"[{e['paso']:02d}] {e['tipo']} | {extras}")